# pretty-missing -- Feature Demo

This notebook walks through all major features of `missing_matrix()` and `missing_matrix_html()`, one parameter at a time.

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy", "matplotlib", "scipy", "pandas", "plotly",
])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from pretty_missing import missing_matrix, missing_matrix_html

# Load the toy RNA-Seq dataset (80 genes x 30 samples, 3-level MultiIndex columns)
df = pd.read_csv(Path("data/toy_rnaseq.csv"), index_col=0, header=[0, 1, 2])
print(f"Shape: {df.shape[0]} genes x {df.shape[1]} samples")
print(f"Column levels: {df.columns.names}")
print(f"Overall missingness: {df.isnull().sum().sum() / df.size:.1%}")
df.head(3)

## 1. Basic usage (defaults only)

Just pass a DataFrame. With MultiIndex columns, annotation strips and dendrogram appear automatically.

In [ ]:
fig = missing_matrix(df)
plt.show()

## 2. Title, subtitle, and metadata

Add a bold title and an italic subtitle line for dataset context.

In [ ]:
n_genes, n_samples = df.shape
overall = df.isnull().sum().sum() / df.size

fig = missing_matrix(
    df,
    title="Gene Detection Matrix -- RNA-Seq QC",
    subtitle=f"{n_genes} genes x {n_samples} samples | {overall:.0%} missing overall",
)
plt.show()

## 3. Custom annotation colors

Override the default palette for specific annotation levels. Keys can be level indices (int) or level names (str).

In [ ]:
fig = missing_matrix(
    df,
    title="Custom Annotation Colors",
    annotation_colors={
        "Medium_Type": {"Fresh": "#88CCEE", "Conditioned": "#CC6677"},
        "Medium_Condition": {"SF": "#44AA99", "FBS": "#DDCC77", "AS": "#AA4499"},
    },
)
plt.show()

## 4. Completeness sparkline variants

### 4a. Below (default) with threshold line

`completeness="below"` shows per-sample detection rate under the matrix. Add a dashed red threshold line with `completeness_threshold`.

In [ ]:
fig = missing_matrix(
    df,
    title="Completeness Below + Threshold",
    completeness="below",
    completeness_threshold=0.5,
)
plt.show()

### 4b. Side sparkline

`completeness="side"` shows per-gene detection rate to the right of the matrix.

In [ ]:
fig = missing_matrix(
    df,
    title="Completeness Side",
    completeness="side",
    completeness_threshold=0.7,
)
plt.show()

## 5. Legend placement

Move legends to any corner with `legend_loc`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 8))
plt.close(fig)

fig1 = missing_matrix(df, title="Legend: upper left", legend_loc="upper left")
fig2 = missing_matrix(df, title="Legend: lower right", legend_loc="lower right")
plt.show()

## 6. Clustering and sorting options

### 6a. No clustering, no gene sorting

Disable sample clustering and gene sorting to show the original data order.

In [ ]:
fig = missing_matrix(
    df,
    title="No clustering, original order",
    cluster_samples=False,
    sort_genes=None,
    show_dendrogram=False,
)
plt.show()

### 6b. Ascending gene sort + hidden dendrogram

Sort genes so least-complete genes appear at the top. Hide the dendrogram while keeping clustering.

In [ ]:
fig = missing_matrix(
    df,
    title="Ascending gene sort, no dendrogram",
    sort_genes="ascending",
    show_dendrogram=False,
)
plt.show()

## 7. Split by factor

`split_by` renders one independently-clustered panel per factor level, side by side. The split level is automatically removed from annotations.

In [ ]:
fig = missing_matrix(
    df,
    title="Split by Medium_Condition",
    split_by="Medium_Condition",
    annotation_levels=[0],
)
plt.show()

## 8. Custom matrix colors and font sizes

Change the detected/missing colours and fine-tune font sizes independently.

In [ ]:
fig = missing_matrix(
    df,
    title="Custom Colors + Font Sizes",
    color_present="#1a5276",
    color_missing="#fadbd8",
    fontsize=11,
    fontsize_rows=5,
    fontsize_cols=7,
    fontsize_legend=9,
    fontsize_annotations=10,
)
plt.show()

## 9. Group completeness summary

`group_summary` prints a per-group completeness table to the console. Pass a level name or index.

In [ ]:
fig = missing_matrix(
    df,
    title="With Group Summary",
    group_summary="Medium_Condition",
    show_dendrogram=False,
)
plt.show()

## 10. Plain DataFrame (no MultiIndex)

Works with simple column names too -- no annotation strips, just the matrix + clustering.

In [ ]:
# Flatten to plain columns
df_plain = df.copy()
df_plain.columns = [f"{mt}_{mc}_{oid}" for mt, mc, oid in df.columns]

fig = missing_matrix(
    df_plain,
    title="Plain DataFrame (no MultiIndex)",
)
plt.show()

## 11. Interactive HTML export

`missing_matrix_html()` creates a Plotly-based interactive version with hover tooltips showing gene, sample, annotation levels, and detection status.

In [ ]:
from IPython.display import HTML, display

html = missing_matrix_html(
    df,
    title="Gene Detection Matrix (Interactive)",
    subtitle=f"{n_genes} genes x {n_samples} samples",
    completeness_threshold=0.5,
)
display(HTML(html))